In [1]:
import pandas as pd
import os
import uuid


### merge CSV files containing specific skill additions into one file

In [ ]:
combined = pd.concat(
    [pd.read_csv("data/skills_cs.csv"),
     pd.read_csv("data/topic_collections/digCompSkillsCollection_cs.csv"),
     pd.read_csv("data/topic_collections/digitalSkillsCollection_cs.csv"),
     pd.read_csv("data/topic_collections/greenSkillsCollection_cs.csv"),
     pd.read_csv("data/topic_collections/languageSkillsCollection_cs.csv"),
     pd.read_csv("data/topic_collections/researchSkillsCollection_cs.csv"),
     pd.read_csv("data/topic_collections/transversalSkillsCollection_cs.csv")
     ],
    ignore_index=True
)

combined.to_csv("data/combinedSkills_cs.csv", index=False)

combined_occ = pd.concat(
    [pd.read_csv("data/occupations_cs.csv"),
     pd.read_csv("data/topic_collections/researchOccupationsCollection_cs.csv")],
    ignore_index=True
)
combined_occ.to_csv("data/combinedOccupations_cs.csv",index=False)

### Util functions
- replace URLs used in ESCO datasets with UUIDs
- format incosistently split skill altlabels (eg pipe, tab, newline) and store as list

In [2]:
uri_to_uuid_map = {}


def store_uri(uri):
    key = f"key_{len(uri_to_uuid_map) + 1}"
    uri_to_uuid_map[uri.strip().lower()] = key
    return key


def get_uuid_from_concept_uri(concept_uri):
    return uri_to_uuid_map.get(concept_uri.strip().lower())

def format_alt_labels(alts_str):
    a = (
    alts_str
        .replace("\xa0", " ")
        .replace("|", "\n")
        .replace("0.0", "")
    )
    alts = "\n".join(x.strip() for x in a.splitlines())

    if alts == "nan":
        return ""
    return alts


### define record formats
= formats of data to work with without extra relations

_ISCO == international skill groups (o*net)_

In [3]:
def isco_groups_transformer(record):
    return {
        "ESCOURI": record["conceptUri"],
        "ID": store_uri(record['conceptUri']),
        "UUIDHISTORY": str(uuid.uuid4()),
        "CODE": record["code"],
        "PREFERREDLABEL": record["preferredLabel"],
        "ALTLABELS": record["altLabels"], # currently no alt labeling exists in the czech dataset,so no need to format here for now
        "DESCRIPTION": record["description"],
    }


def skills_transformer(record):
    return {
        "ESCOURI": record["conceptUri"],
        "ID": store_uri(record['conceptUri']),
        "UUIDHISTORY": str(uuid.uuid4()),
        "SKILLTYPE": record.get("skillType"),
        "REUSELEVEL": record.get("reuseLevel"),
        "PREFERREDLABEL": record["preferredLabel"],
        "ALTLABELS": format_alt_labels(str(record["altLabels"])),
        "DESCRIPTION": record["description"],
        "DEFINITION": record.get("definition"),
        "SCOPENOTE": record.get("scopeNote"),
    }


def occupations_transformer(record):
    return {
        "ESCOURI": record["conceptUri"],
        "ID": store_uri(record["conceptUri"]),
        "UUIDHISTORY": str(uuid.uuid4()),
        "ISCOGROUPCODE": record["iscoGroup"],
        "CODE": record["code"],
        "PREFERREDLABEL": record["preferredLabel"],
        "ALTLABELS": format_alt_labels(str(record["altLabels"])),
        "DESCRIPTION": record["description"],
        "DEFINITION": record["definition"],
        "SCOPENOTE": record.get("scopeNote"),
        "REGULATEDPROFESSIONNOTE": record.get("regulatedProfessionNote"),
        "OCCUPATIONTYPE": "ESCO",
    }


def skill_groups_transformer(record):
    return {
        "ESCOURI": record["conceptUri"],
        "ID": store_uri(record['conceptUri']),
        "UUIDHISTORY": str(uuid.uuid4()),
        "CODE": record["code"],
        "PREFERREDLABEL": record["preferredLabel"],
        "ALTLABELS": record["altLabels"],
        "DESCRIPTION": record["description"],
        "SCOPENOTE": record.get("scopeNote"),
    }


def skills_skills_relations_transformer(record):
    requiring_id = get_uuid_from_concept_uri(record['originalSkillUri'])
    required_id = get_uuid_from_concept_uri(record['relatedSkillUri'])

    return {
        "REQUIRINGID": requiring_id,
        "RELATIONTYPE": record['relationType'],
        "REQUIREDID": required_id
    }


def occupations_skills_relations_transformer(record):
    occupation_id = get_uuid_from_concept_uri(record['occupationUri'])
    skill_id = get_uuid_from_concept_uri(record['skillUri'])
    occupation_type = record.get('occupationType', "ESCO")

    return {
        "OCCUPATIONTYPE": occupation_type,
        "OCCUPATIONID": occupation_id,
        "RELATIONTYPE": record['relationType'],
        "SKILLID": skill_id
    }


def occupation_skill_hierarchy_transformer(record):
    parent_id = get_uuid_from_concept_uri(record['broaderUri'])
    child_id = get_uuid_from_concept_uri(record['conceptUri'])

    return {
        "PARENTOBJECTTYPE": record['broaderType'],
        "PARENTID": parent_id,
        "CHILDID": child_id,
        "CHILDOBJECTTYPE": record['conceptType']
    }


### Reformat

In [4]:

def export_all(filepath, outputs, templates):
    for filename, record in templates:
        inputf = os.path.join(filepath, filename)
        output = os.path.join(outputs, filename)
        df = pd.read_csv(inputf)
        transformed = df.apply(record, axis=1)
        transformed = pd.DataFrame(transformed.tolist())
        transformed.to_csv(output, index=False)


template_map = [
    ('ISCOGroups_cs.csv', isco_groups_transformer),
    ('combinedOccupations_cs.csv', occupations_transformer),
    ('combinedSkills_cs.csv', skills_transformer),
    ('skillGroups_cs.csv', skill_groups_transformer),
    ('broaderRelationsOccPillar_cs.csv', occupation_skill_hierarchy_transformer),
    ('broaderRelationsSkillPillar_cs.csv', occupation_skill_hierarchy_transformer),
    ('occupationSkillRelations_cs.csv', occupations_skills_relations_transformer),
    ('skillSkillRelations_cs.csv', skills_skills_relations_transformer)
]

In [5]:
data_dir = os.path.join(os.getcwd(), "data")
out_dir = os.path.join(os.getcwd(), "formatted")
os.makedirs(out_dir, exist_ok=True)

export_all(data_dir, out_dir, template_map)

In [2]:
origin = pd.DataFrame(pd.read_csv('formatted/combinedOccupations_cs.csv'))
split_titles = origin.copy()
split_titles['occupation'] = (
    split_titles['occupations']
        .fillna('')
        .str.split('\n')
)
split_titles = split_titles.explode('occupation')
split_titles['occupation'] = split_titles['occupation'].str.strip()
split_titles = split_titles[split_titles['occupation'] != '']
split_titles[['occupation', 'preffered_label', 'esco_code', 'uuid']].to_csv('formatted/occupations_split_cs.csv')

KeyError: 'occupations'